# Deep Generative Models - Assignment 2
## Masked Autoregressive Flow (MAF) & CycleGAN

**Student Name**: [Your Name]  
**Student ID**: [Your ID]  

This notebook contains complete implementations for:
- **Part 1**: Masked Autoregressive Flow (MAF) for density estimation and anomaly detection
- **Part 2**: CycleGAN for unpaired image-to-image translation

---

### Table of Contents
1. [MAF Implementation](#maf)
   - MADE (Masked Autoencoder)
   - MAF Blocks
   - Training & Generation
   - Anomaly Detection
   
2. [CycleGAN Implementation](#cyclegan)
   - Generator (ResNet-based)
   - Discriminator (PatchGAN)
   - Loss Functions
   - Training Loop
   
3. [Experiments & Results](#results)
   - MAF Training on Capsule Dataset
   - Anomaly Detection Evaluation
   - CycleGAN Training
   - Qualitative & Quantitative Analysis

---


## Part 1: Masked Autoregressive Flow (MAF) - Section 1

### Implementation Details

**Architecture Specifications:**
- **Input Dimensions**: 128 × 128 × 3 = 49,152
- **MADE Architecture**:
  - Input layer: 49,152 → 512
  - Hidden layer: 512 → 512  
  - Output layer: 512 → 98,304 (2× input for scale and translation parameters)
- **Number of MAF Blocks**: 7
- **Batch Size**: 3
- **Epochs**: 100
- **Learning Rate**: 0.0001
- **Optimizer**: Adam

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class MaskedLinear(nn.Module):
    def __init__(self, in_features, out_features, mask):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        # Register the mask as a buffer (not trained)
        self.register_buffer('mask', mask) 

    def forward(self, x):
        # Apply mask to the weight matrix
        masked_weight = self.linear.weight * self.mask
        return F.linear(x, masked_weight, self.linear.bias)

def create_masks(input_dim, hidden_dims, output_dim):
    """
    Create masks for MADE to ensure autoregressive property.
    Each output dimension i should only depend on input dimensions < i.
    """
    masks = []
    
    # Assign degrees to input units (1 to D)
    m_input = torch.arange(1, input_dim + 1)
    
    # For hidden layers, assign random degrees between min(m_input) and max(m_input)-1
    m_hidden = []
    for hidden_dim in hidden_dims:
        m_h = torch.randint(low=1, high=input_dim, size=(hidden_dim,))
        m_hidden.append(m_h)
    
    # For output layer, degrees are 1 to D (each output corresponds to one input dimension)
    m_output = torch.arange(1, output_dim // 2 + 1).repeat(2)  # For both scale and translation
    
    # Create mask from input to first hidden layer
    mask = (m_input.unsqueeze(1) <= m_hidden[0].unsqueeze(0)).float()
    masks.append(mask)
    
    # Create masks between hidden layers
    for i in range(len(hidden_dims) - 1):
        mask = (m_hidden[i].unsqueeze(1) <= m_hidden[i+1].unsqueeze(0)).float()
        masks.append(mask)
    
    # Create mask from last hidden layer to output
    mask = (m_hidden[-1].unsqueeze(1) < m_output.unsqueeze(0)).float()
    masks.append(mask)
    
    return masks

class MADE(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # Create masks
        masks = create_masks(input_dim, hidden_dims, output_dim)
        
        # Build masked layers
        self.layers = nn.ModuleList()
        dims = [input_dim] + hidden_dims + [output_dim]
        
        for i in range(len(dims) - 1):
            self.layers.append(MaskedLinear(dims[i], dims[i+1], masks[i]))
    
    def forward(self, x):
        # Flatten input if needed
        batch_size = x.shape[0]
        x = x.view(batch_size, -1)
        
        # Pass through masked layers
        for i, layer in enumerate(self.layers[:-1]):
            x = F.relu(layer(x))
        
        # Last layer (no activation)
        x = self.layers[-1](x)
        
        return x


In [ ]:
class MAFBlock(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 512]):
        super().__init__()
        self.input_dim = input_dim
        # MADE outputs both scale and translation parameters (2 * input_dim)
        self.made = MADE(input_dim, hidden_dims, 2 * input_dim)

    def forward(self, x):
        """
        Forward pass: z = (x - t) / s
        Returns z and log determinant of Jacobian
        """
        batch_size = x.shape[0]
        x_flat = x.view(batch_size, -1)
        
        # Get scale and translation from MADE
        s_and_t = self.made(x_flat)
        s, t = s_and_t.chunk(2, dim=1)
        
        # Use sigmoid for stability and ensure s > 0
        s = torch.sigmoid(s + 2.0)  # Bias towards s ≈ 1
        
        # Apply transformation
        z = (x_flat - t) / (s + 1e-8)
        
        # Log-determinant of Jacobian: log|det(dz/dx)| = -sum(log(s))
        log_det_J = -torch.sum(torch.log(s + 1e-8), dim=1)
        
        return z, log_det_J

    def inverse(self, z):
        """
        Inverse pass: x = s * z + t
        This is slow because it requires autoregressive computation
        """
        batch_size = z.shape[0]
        x = torch.zeros_like(z)
        
        # Autoregressive generation (slow!)
        for i in range(self.input_dim):
            # Get parameters for dimension i based on x[0:i]
            s_and_t = self.made(x)
            s, t = s_and_t.chunk(2, dim=1)
            s = torch.sigmoid(s + 2.0)
            
            # Update dimension i
            x[:, i] = s[:, i] * z[:, i] + t[:, i]
        
        return x


In [ ]:
class MAF(nn.Module):
    def __init__(self, input_dim, num_blocks=7, hidden_dims=[512, 512]):
        super().__init__()
        self.input_dim = input_dim
        self.blocks = nn.ModuleList([
            MAFBlock(input_dim, hidden_dims) for _ in range(num_blocks)
        ])
        
        # Base distribution (Standard Normal)
        self.register_buffer('base_mean', torch.zeros(input_dim))
        self.register_buffer('base_std', torch.ones(input_dim))

    def forward(self, x):
        """
        Calculate negative log likelihood (NLL)
        Returns z and log probability
        """
        batch_size = x.shape[0]
        x_flat = x.view(batch_size, -1)
        
        log_prob = torch.zeros(batch_size, device=x.device)
        
        # Pass through all blocks
        z = x_flat
        for block in self.blocks:
            z, log_det_J = block(z)
            log_prob += log_det_J
        
        # Add log probability under base distribution (Standard Normal)
        log_prob_base = -0.5 * (z ** 2 + np.log(2 * np.pi)).sum(dim=1)
        log_prob += log_prob_base
        
        return z, log_prob
    
    def calculate_nll(self, x):
        """Calculate negative log likelihood"""
        _, log_prob = self.forward(x)
        return -log_prob.mean()
        
    def generate(self, num_samples, device='cpu'):
        """
        Generate samples by sampling from base distribution and 
        passing through inverse transforms (slow!)
        """
        # Sample from base distribution
        z = torch.randn(num_samples, self.input_dim, device=device)
        
        # Pass through inverse transforms in reverse order
        for block in reversed(self.blocks):
            z = block.inverse(z)
        
        return z


In [ ]:
# MAF Training and Evaluation
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
from tqdm import tqdm
import time
import matplotlib.pyplot as plt

class CapsuleDataset(Dataset):
    def __init__(self, root_dir, transform=None, img_size=128):
        self.root_dir = root_dir
        self.transform = transform
        self.img_size = img_size
        self.images = []
        
        # Load all image paths
        for fname in os.listdir(root_dir):
            if fname.endswith(('.png', '.jpg', '.jpeg')):
                self.images.append(os.path.join(root_dir, fname))
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

def train_maf(model, train_loader, num_epochs=100, lr=0.0001, device='cpu'):
    """Train MAF model"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in pbar:
            batch = batch.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            nll = model.calculate_nll(batch)
            
            # Backward pass
            nll.backward()
            optimizer.step()
            
            epoch_loss += nll.item()
            pbar.set_postfix({'NLL': nll.item()})
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}, Average NLL: {avg_loss:.4f}')
    
    return losses

def generate_images_maf(model, num_images=5, img_size=128, device='cpu'):
    """Generate images using MAF (slow due to autoregressive inverse)"""
    model.eval()
    
    start_time = time.time()
    
    with torch.no_grad():
        samples = model.generate(num_images, device=device)
        samples = samples.view(num_images, 3, img_size, img_size)
        samples = torch.clamp(samples, -1, 1)
        samples = (samples + 1) / 2  # Denormalize to [0, 1]
    
    generation_time = time.time() - start_time
    
    return samples, generation_time

# Setup transforms
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])


In [ ]:
### CycleGAN Implementation ###

class ResidualBlock(nn.Module):
    """ResNet residual block with reflection padding"""
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels)
        )
    
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    """CycleGAN Generator with ResNet blocks"""
    def __init__(self, input_nc=3, output_nc=3, ngf=64, num_residual_blocks=9):
        super().__init__()
        
        # Initial convolution: C7S1-64
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, stride=1, padding=0),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True)
        ]
        
        # Downsampling: D128, D256
        in_channels = ngf
        out_channels = ngf * 2
        for i in range(2):
            model += [
                nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ]
            in_channels = out_channels
            out_channels = out_channels * 2
        
        # Residual blocks: R256
        for i in range(num_residual_blocks):
            model += [ResidualBlock(in_channels)]
        
        # Upsampling: U128, U64
        out_channels = in_channels // 2
        for i in range(2):
            model += [
                nn.ConvTranspose2d(in_channels, out_channels, 
                                   kernel_size=3, stride=2, 
                                   padding=1, output_padding=1),
                nn.InstanceNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ]
            in_channels = out_channels
            out_channels = out_channels // 2
        
        # Output convolution: C7S1-3
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, output_nc, kernel_size=7, stride=1, padding=0),
            nn.Tanh()
        ]
        
        self.model = nn.Sequential(*model)
    
    def forward(self, x):
        return self.model(x)


In [ ]:
class Discriminator(nn.Module):
    """PatchGAN Discriminator (70x70 patches)"""
    def __init__(self, input_nc=3, ndf=64):
        super().__init__()
        
        # C64: No normalization on first layer
        model = [
            nn.Conv2d(input_nc, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        # C128
        model += [
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        # C256
        model += [
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        # C512
        model += [
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=1, padding=1),
            nn.InstanceNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True)
        ]
        
        # Output layer (patch predictions)
        model += [
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1)
        ]
        
        self.model = nn.Sequential(*model)
    
    def forward(self, x):
        return self.model(x)


In [ ]:
# CycleGAN Loss Functions

# Hyperparameters
lambda_A = 10.0  # Cycle consistency loss weight for A->B->A
lambda_B = 10.0  # Cycle consistency loss weight for B->A->B
lambda_identity = 0.5  # Identity loss weight

def adversarial_loss(prediction, is_real):
    """
    Adversarial loss using MSE (LSGAN)
    Target: 1.0 for real, 0.0 for fake
    """
    if is_real:
        target = torch.ones_like(prediction)
    else:
        target = torch.zeros_like(prediction)
    return F.mse_loss(prediction, target)

def cycle_consistency_loss(real_image, reconstructed_image):
    """
    Cycle consistency loss: ||F(G(x)) - x||_1
    Uses L1 loss for sharper images
    """
    return F.l1_loss(reconstructed_image, real_image)

def identity_loss(generator, real_image):
    """
    Identity loss: ||G(y) - y||_1
    Generator should not change images from its target domain
    E.g., G_A (horse->zebra) should not change zebra images
    """
    identity_image = generator(real_image)
    return F.l1_loss(identity_image, real_image)

def generator_loss(D, fake_image):
    """
    Generator tries to fool discriminator
    Loss = E[(D(G(x)) - 1)^2]
    """
    pred_fake = D(fake_image)
    return adversarial_loss(pred_fake, True)

def discriminator_loss(D, real_image, fake_image):
    """
    Discriminator tries to distinguish real from fake
    Loss = E[(D(real) - 1)^2] + E[D(fake)^2]
    """
    pred_real = D(real_image)
    pred_fake = D(fake_image.detach())
    
    loss_real = adversarial_loss(pred_real, True)
    loss_fake = adversarial_loss(pred_fake, False)
    
    return (loss_real + loss_fake) * 0.5


In [ ]:
import random
from collections import deque

class ImagePool:
    """
    Image buffer that stores previously generated images.
    Used to update discriminator with history of generated images
    rather than only the latest ones. This helps reduce model oscillation.
    """
    def __init__(self, pool_size=50):
        self.pool_size = pool_size
        self.images = []
    
    def query(self, images):
        """
        Return images from pool.
        With 50% probability, return image from pool and replace it with new image.
        With 50% probability, return the new image.
        """
        if self.pool_size == 0:
            return images
        
        return_images = []
        
        for image in images:
            image = image.unsqueeze(0)
            
            if len(self.images) < self.pool_size:
                # Pool not full, just add and return
                self.images.append(image)
                return_images.append(image)
            else:
                # Pool is full
                if random.uniform(0, 1) > 0.5:
                    # Return random image from pool and replace with new
                    random_id = random.randint(0, self.pool_size - 1)
                    return_images.append(self.images[random_id].clone())
                    self.images[random_id] = image
                else:
                    # Return new image
                    return_images.append(image)
        
        return torch.cat(return_images, dim=0)


In [ ]:
# CycleGAN Training Loop

def train_cyclegan(G_AB, G_BA, D_A, D_B, train_loader_A, train_loader_B, 
                   num_epochs=20, lr=0.0002, beta1=0.5, device='cpu'):
    """
    Train CycleGAN model
    G_AB: Generator A->B (e.g., horse->zebra)
    G_BA: Generator B->A (e.g., zebra->horse)
    D_A: Discriminator for domain A
    D_B: Discriminator for domain B
    """
    
    # Move models to device
    G_AB = G_AB.to(device)
    G_BA = G_BA.to(device)
    D_A = D_A.to(device)
    D_B = D_B.to(device)
    
    # Optimizers
    optimizer_G = optim.Adam(
        list(G_AB.parameters()) + list(G_BA.parameters()),
        lr=lr, betas=(beta1, 0.999)
    )
    optimizer_D_A = optim.Adam(D_A.parameters(), lr=lr, betas=(beta1, 0.999))
    optimizer_D_B = optim.Adam(D_B.parameters(), lr=lr, betas=(beta1, 0.999))
    
    # Learning rate schedulers (decay after half epochs)
    scheduler_G = optim.lr_scheduler.LambdaLR(
        optimizer_G, lr_lambda=lambda epoch: 1.0 - max(0, epoch - num_epochs // 2) / (num_epochs // 2)
    )
    scheduler_D_A = optim.lr_scheduler.LambdaLR(
        optimizer_D_A, lr_lambda=lambda epoch: 1.0 - max(0, epoch - num_epochs // 2) / (num_epochs // 2)
    )
    scheduler_D_B = optim.lr_scheduler.LambdaLR(
        optimizer_D_B, lr_lambda=lambda epoch: 1.0 - max(0, epoch - num_epochs // 2) / (num_epochs // 2)
    )
    
    # Image pools
    fake_A_pool = ImagePool(pool_size=50)
    fake_B_pool = ImagePool(pool_size=50)
    
    # Track losses
    history = {
        'G_loss': [],
        'D_A_loss': [],
        'D_B_loss': [],
        'cycle_loss': [],
        'identity_loss': []
    }
    
    for epoch in range(num_epochs):
        G_AB.train()
        G_BA.train()
        D_A.train()
        D_B.train()
        
        epoch_G_loss = 0
        epoch_D_A_loss = 0
        epoch_D_B_loss = 0
        epoch_cycle_loss = 0
        epoch_identity_loss = 0
        
        # Iterate through both datasets
        data_iter_A = iter(train_loader_A)
        data_iter_B = iter(train_loader_B)
        
        num_batches = min(len(train_loader_A), len(train_loader_B))
        pbar = tqdm(range(num_batches), desc=f'Epoch {epoch+1}/{num_epochs}')
        
        for i in pbar:
            try:
                real_A = next(data_iter_A).to(device)
                real_B = next(data_iter_B).to(device)
            except StopIteration:
                break
            
            batch_size = min(real_A.size(0), real_B.size(0))
            real_A = real_A[:batch_size]
            real_B = real_B[:batch_size]
            
            # ==================== Train Generators ==================== #
            optimizer_G.zero_grad()
            
            # Identity loss
            loss_identity_A = identity_loss(G_BA, real_A) * lambda_A * lambda_identity
            loss_identity_B = identity_loss(G_AB, real_B) * lambda_B * lambda_identity
            loss_identity_total = loss_identity_A + loss_identity_B
            
            # GAN loss
            fake_B = G_AB(real_A)
            loss_GAN_AB = generator_loss(D_B, fake_B)
            
            fake_A = G_BA(real_B)
            loss_GAN_BA = generator_loss(D_A, fake_A)
            
            # Cycle consistency loss
            recovered_A = G_BA(fake_B)
            loss_cycle_A = cycle_consistency_loss(real_A, recovered_A) * lambda_A
            
            recovered_B = G_AB(fake_A)
            loss_cycle_B = cycle_consistency_loss(real_B, recovered_B) * lambda_B
            
            loss_cycle_total = loss_cycle_A + loss_cycle_B
            
            # Total generator loss
            loss_G = loss_GAN_AB + loss_GAN_BA + loss_cycle_total + loss_identity_total
            
            loss_G.backward()
            optimizer_G.step()
            
            # ==================== Train Discriminator A ==================== #
            optimizer_D_A.zero_grad()
            
            fake_A_pooled = fake_A_pool.query(fake_A.detach())
            loss_D_A = discriminator_loss(D_A, real_A, fake_A_pooled)
            
            loss_D_A.backward()
            optimizer_D_A.step()
            
            # ==================== Train Discriminator B ==================== #
            optimizer_D_B.zero_grad()
            
            fake_B_pooled = fake_B_pool.query(fake_B.detach())
            loss_D_B = discriminator_loss(D_B, real_B, fake_B_pooled)
            
            loss_D_B.backward()
            optimizer_D_B.step()
            
            # Track losses
            epoch_G_loss += loss_G.item()
            epoch_D_A_loss += loss_D_A.item()
            epoch_D_B_loss += loss_D_B.item()
            epoch_cycle_loss += loss_cycle_total.item()
            epoch_identity_loss += loss_identity_total.item()
            
            pbar.set_postfix({
                'G': f'{loss_G.item():.3f}',
                'D_A': f'{loss_D_A.item():.3f}',
                'D_B': f'{loss_D_B.item():.3f}'
            })
        
        # Update learning rates
        scheduler_G.step()
        scheduler_D_A.step()
        scheduler_D_B.step()
        
        # Save epoch statistics
        history['G_loss'].append(epoch_G_loss / num_batches)
        history['D_A_loss'].append(epoch_D_A_loss / num_batches)
        history['D_B_loss'].append(epoch_D_B_loss / num_batches)
        history['cycle_loss'].append(epoch_cycle_loss / num_batches)
        history['identity_loss'].append(epoch_identity_loss / num_batches)
        
        print(f'\nEpoch {epoch+1} - G: {history["G_loss"][-1]:.4f}, '
              f'D_A: {history["D_A_loss"][-1]:.4f}, D_B: {history["D_B_loss"][-1]:.4f}, '
              f'Cycle: {history["cycle_loss"][-1]:.4f}, Identity: {history["identity_loss"][-1]:.4f}')
    
    return history


In [ ]:
# Data Loading and Preprocessing for CycleGAN

class ImageDataset(Dataset):
    """Dataset for loading unpaired images from two domains"""
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        
        # Load all image paths
        for fname in os.listdir(root_dir):
            if fname.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                self.images.append(os.path.join(root_dir, fname))
        
        print(f"Loaded {len(self.images)} images from {root_dir}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx % len(self.images)]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

# Transform for CycleGAN
cyclegan_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


In [ ]:
# Visualization and Evaluation Functions

def visualize_samples(images, titles=None, figsize=(15, 5)):
    """Visualize a batch of images"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    
    if n == 1:
        axes = [axes]
    
    for i, (img, ax) in enumerate(zip(images, axes)):
        if isinstance(img, torch.Tensor):
            img = img.cpu().detach()
            if img.dim() == 4:
                img = img[0]
            img = img.permute(1, 2, 0).numpy()
            img = (img + 1) / 2  # Denormalize from [-1, 1] to [0, 1]
            img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.axis('off')
        if titles and i < len(titles):
            ax.set_title(titles[i])
    
    plt.tight_layout()
    plt.show()

def test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device='cpu', num_samples=5):
    """Test CycleGAN and visualize results"""
    G_AB.eval()
    G_BA.eval()
    
    with torch.no_grad():
        # Get samples from domain A
        real_A = next(iter(test_loader_A))[:num_samples].to(device)
        fake_B = G_AB(real_A)
        recovered_A = G_BA(fake_B)
        
        # Get samples from domain B
        real_B = next(iter(test_loader_B))[:num_samples].to(device)
        fake_A = G_BA(real_B)
        recovered_B = G_AB(fake_A)
    
    # Visualize A -> B -> A
    for i in range(num_samples):
        visualize_samples(
            [real_A[i], fake_B[i], recovered_A[i]],
            titles=['Real A', 'Fake B', 'Recovered A']
        )
    
    # Visualize B -> A -> B
    for i in range(num_samples):
        visualize_samples(
            [real_B[i], fake_A[i], recovered_B[i]],
            titles=['Real B', 'Fake A', 'Recovered B']
        )

def plot_training_history(history):
    """Plot training losses"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Generator loss
    axes[0, 0].plot(history['G_loss'])
    axes[0, 0].set_title('Generator Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True)
    
    # Discriminator losses
    axes[0, 1].plot(history['D_A_loss'], label='D_A')
    axes[0, 1].plot(history['D_B_loss'], label='D_B')
    axes[0, 1].set_title('Discriminator Losses')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Cycle consistency loss
    axes[1, 0].plot(history['cycle_loss'])
    axes[1, 0].set_title('Cycle Consistency Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].grid(True)
    
    # Identity loss
    axes[1, 1].plot(history['identity_loss'])
    axes[1, 1].set_title('Identity Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Anomaly Detection using MAF

from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

def calculate_anomaly_scores(model, test_loader, device='cpu'):
    """
    Calculate anomaly scores for test images using MAF.
    Anomaly score = Negative Log Likelihood (NLL)
    Higher NLL indicates the image is less likely under the learned distribution
    """
    model.eval()
    anomaly_scores = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Computing anomaly scores'):
            batch = batch.to(device)
            _, log_prob = model.forward(batch)
            nll = -log_prob  # Negative log likelihood as anomaly score
            anomaly_scores.extend(nll.cpu().numpy())
    
    return np.array(anomaly_scores)

def evaluate_anomaly_detection(normal_scores, anomaly_scores):
    """
    Evaluate anomaly detection performance using AUROC
    
    Args:
        normal_scores: Anomaly scores for normal images (lower is better)
        anomaly_scores: Anomaly scores for anomalous images (higher is better)
    
    Returns:
        auroc: Area Under ROC Curve
        fpr: False Positive Rate
        tpr: True Positive Rate
    """
    # Create labels: 0 for normal, 1 for anomaly
    y_true = np.concatenate([
        np.zeros(len(normal_scores)),
        np.ones(len(anomaly_scores))
    ])
    
    # Combine scores
    y_scores = np.concatenate([normal_scores, anomaly_scores])
    
    # Calculate AUROC
    auroc = roc_auc_score(y_true, y_scores)
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    
    return auroc, fpr, tpr

def plot_roc_curve(fpr, tpr, auroc):
    """Plot ROC curve"""
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUROC = {auroc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Anomaly Detection')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_score_distributions(normal_scores, anomaly_scores):
    """Plot distributions of anomaly scores"""
    plt.figure(figsize=(10, 6))
    plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal', density=True)
    plt.hist(anomaly_scores, bins=50, alpha=0.6, label='Anomaly', density=True)
    plt.xlabel('Anomaly Score (NLL)')
    plt.ylabel('Density')
    plt.title('Distribution of Anomaly Scores')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# ==============================================================================
# PART 1: MAF Training and Evaluation Example
# ==============================================================================

"""
Example usage for MAF training on Capsule dataset

# 1. Download and prepare data
!wget https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420937454-1629951595/capsule.tar.xz
!tar -xf capsule.tar.xz

# 2. Setup dataset and dataloader
train_dataset = CapsuleDataset(
    root_dir='capsule/train/good',
    transform=transform,
    img_size=128
)

train_loader = DataLoader(
    train_dataset,
    batch_size=3,
    shuffle=True,
    num_workers=2
)

# 3. Initialize MAF model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = 128 * 128 * 3  # 49152
maf_model = MAF(
    input_dim=input_dim,
    num_blocks=7,
    hidden_dims=[512, 512]
)

print(f"Model parameters: {sum(p.numel() for p in maf_model.parameters()):,}")

# 4. Train the model
losses = train_maf(
    model=maf_model,
    train_loader=train_loader,
    num_epochs=100,
    lr=0.0001,
    device=device
)

# 5. Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Negative Log Likelihood')
plt.title('MAF Training Loss')
plt.grid(True)
plt.show()

# 6. Generate images (slow!)
print("Generating 5 images...")
generated_images, gen_time = generate_images_maf(
    model=maf_model,
    num_images=5,
    img_size=128,
    device=device
)

print(f"Generation time: {gen_time:.2f} seconds ({gen_time/5:.2f} sec per image)")

# 7. Visualize generated images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    img = generated_images[i].permute(1, 2, 0).cpu().numpy()
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()
"""


## Part 1: MAF - Section 2: Training and Generation

### Question Answers:

#### Q1: Why is generation slow in autoregressive models?

**Answer**: In autoregressive models like MAF, generation is slow because of the **sequential dependency** in the inverse transformation:

1. **Forward Pass (Training)**: Fast and parallelizable
   - Given input x, compute z = (x - t(x)) / s(x)
   - All dimensions can be computed in parallel since MADE can process the entire input at once
   
2. **Inverse Pass (Generation)**: Slow and sequential
   - To generate x from z, we need: x = s(x) · z + t(x)
   - **Problem**: Both s and t depend on x itself!
   - Must compute dimension by dimension:
     ```
     for i in 1 to D:
         x[i] = s[i](x[1:i-1]) · z[i] + t[i](x[1:i-1])
     ```
   - Each dimension requires a full forward pass through MADE
   - For 49,152 dimensions, this means 49,152 sequential MADE evaluations!

**Time Complexity**:
- Forward: O(1) - single pass
- Inverse: O(D) - D sequential passes where D is dimensionality

This is why generating 5 images can take several minutes!

---

#### Q2: How does Inverse Autoregressive Flow (IAF) solve this problem?

**Answer**: IAF reverses the direction of the autoregressive dependency:

**IAF Forward (Generation)**:
- z → x: **x = s(z) · z + t(z)**  [FAST - parallel]
- Parameters depend on z, not x
- All dimensions computed in one pass

**IAF Inverse (Training)**:
- x → z: **z = (x - t(z)) / s(z)**  [SLOW - sequential]
- Need to solve for z iteratively

**Comparison**:
| Model | Training | Generation |
|-------|----------|------------|
| MAF   | Fast ✅   | Slow ❌     |
| IAF   | Slow ❌   | Fast ✅     |

**When to use**:
- **MAF**: When you need fast training (e.g., density estimation, anomaly detection)
- **IAF**: When you need fast generation (e.g., VAE posterior, generative models)

---

In [ ]:
# ==============================================================================
# PART 2: Anomaly Detection with MAF
# ==============================================================================

"""
Example usage for anomaly detection with trained MAF

# 1. Prepare test datasets (normal and anomaly)
test_normal_dataset = CapsuleDataset(
    root_dir='capsule/test/good',
    transform=transform,
    img_size=128
)

test_anomaly_dataset = CapsuleDataset(
    root_dir='capsule/test/crack',  # or other defect types
    transform=transform,
    img_size=128
)

test_normal_loader = DataLoader(test_normal_dataset, batch_size=8, shuffle=False)
test_anomaly_loader = DataLoader(test_anomaly_dataset, batch_size=8, shuffle=False)

# 2. Calculate anomaly scores
print("Calculating anomaly scores for normal test images...")
normal_scores = calculate_anomaly_scores(maf_model, test_normal_loader, device)

print("Calculating anomaly scores for anomalous test images...")
anomaly_scores = calculate_anomaly_scores(maf_model, test_anomaly_loader, device)

# 3. Evaluate performance
auroc, fpr, tpr = evaluate_anomaly_detection(normal_scores, anomaly_scores)
print(f"AUROC: {auroc:.4f}")

# 4. Plot results
plot_roc_curve(fpr, tpr, auroc)
plot_score_distributions(normal_scores, anomaly_scores)

# 5. Show some examples
print(f"Normal images - Mean score: {normal_scores.mean():.4f}, Std: {normal_scores.std():.4f}")
print(f"Anomaly images - Mean score: {anomaly_scores.mean():.4f}, Std: {anomaly_scores.std():.4f}")
"""


## Part 1: MAF - Section 3: Anomaly Detection

### Question Answers:

#### Q1: Why don't we use accuracy as an evaluation metric for anomaly detection?

**Answer**: Accuracy is misleading in anomaly detection due to **severe class imbalance**:

**Example Scenario**:
- Normal samples: 95%
- Anomalies: 5%

A naive model that predicts "normal" for everything achieves **95% accuracy** but detects **0% of anomalies**! This is useless for anomaly detection.

**Why This Happens**:
- Anomalies are rare by definition
- The cost of missing an anomaly (false negative) is usually much higher than a false alarm (false positive)
- Accuracy treats all errors equally, ignoring this asymmetry

**Better Metrics**:
- **AUROC (Area Under ROC Curve)**: Measures performance across all thresholds
- **Precision-Recall AUC**: Better for highly imbalanced data
- **F1-Score**: Balances precision and recall
- **True Positive Rate at fixed False Positive Rate**: Domain-specific requirements

---

#### Q2: What is the concept of anomaly score?

**Answer**: An **anomaly score** is a scalar value that quantifies how "unusual" or "abnormal" a data point is compared to the normal distribution.

**Key Properties**:
- **Higher score → More anomalous**
- **Lower score → More normal**
- Provides a ranking rather than binary classification
- Allows flexible threshold selection based on use case

**Common Anomaly Scores**:

1. **Reconstruction Error** (VAE, Autoencoder):
   ```
   score = ||x - x_reconstructed||
   ```
   - Normal samples reconstruct well (low error)
   - Anomalies reconstruct poorly (high error)

2. **Negative Log-Likelihood** (Normalizing Flows, GMM):
   ```
   score = -log p(x)
   ```
   - Normal samples have high probability (low NLL)
   - Anomalies have low probability (high NLL)

3. **Distance-based** (One-Class SVM, Isolation Forest):
   ```
   score = distance_to_decision_boundary(x)
   ```

**Threshold Selection**:
- Set based on desired false positive rate
- Domain-specific requirements (e.g., medical: minimize false negatives)

---

#### Q3: How can normalizing flows be used for anomaly detection?

**Answer**: Normalizing flows are **excellent** for anomaly detection because they learn the **exact probability density** p(x).

**Method**:
1. **Training**: Learn the distribution of normal data
   - Train flow model on normal samples only
   - Model learns p(x) for normal distribution

2. **Anomaly Scoring**: Use negative log-likelihood
   ```python
   score(x) = -log p(x)
   ```
   - Normal samples: High p(x) → Low score
   - Anomalies: Low p(x) → High score

3. **Detection**: Threshold the scores
   ```python
   is_anomaly = score(x) > threshold
   ```

**Advantages**:
✅ **Exact density estimation** (unlike VAE which has intractable likelihood)
✅ **Principled probabilistic framework**
✅ **No reconstruction needed** (direct likelihood computation)
✅ **Works well for high-dimensional data**

**Why It Works**:
- Normal data lies in high-density regions of the learned distribution
- Anomalies lie in low-density regions (out-of-distribution)
- The likelihood directly measures "typicality"

**Practical Considerations**:
- Requires sufficient normal training data
- Sensitive to distribution shift
- Computational cost for high-dimensional data

---

#### Q4: Can we use normalizing flows for anomaly detection in the same way as VAE reconstruction error?

**Answer**: **Not exactly** - the approaches are fundamentally different, but both can work:

**VAE Approach (Reconstruction-Based)**:
```python
# Training: Learn encoder and decoder
x → encoder → z → decoder → x_reconstructed
loss = reconstruction_error + KL_divergence

# Testing: Measure reconstruction error
anomaly_score = ||x - x_reconstructed||²
```

**Why it works for VAE**:
- Normal samples are well-reconstructed (low error)
- Anomalies are poorly reconstructed (high error)
- Relies on the bottleneck: anomalies can't be encoded/decoded well

**Normalizing Flow Approach (Likelihood-Based)**:
```python
# Training: Learn exact density
x → flow → z (with log|det J|)
loss = -log p(x)

# Testing: Compute likelihood
anomaly_score = -log p(x)
```

**Why it works for Flows**:
- Direct density estimation, no reconstruction
- Anomalies have low probability under learned distribution

**Can we use reconstruction with Flows?**

**Theoretically YES**, but it's not the standard approach:
```python
# Forward: x → z
z, log_prob = flow.forward(x)

# Inverse: z → x_reconstructed
x_reconstructed = flow.inverse(z)

# Anomaly score
score = ||x - x_reconstructed||²
```

**Problems with this approach**:
❌ **Should be perfect reconstruction** (flows are bijective!)
❌ Only fails due to numerical errors, not semantic anomalies
❌ Doesn't leverage the main advantage of flows (exact likelihood)
❌ Much slower (requires expensive inverse pass)

**Conclusion**:
- **VAE**: Use reconstruction error (likelihood intractable)
- **Flows**: Use negative log-likelihood (exact and principled)
- Both work, but they exploit different properties of the models

**Best Practice**:
Stick to likelihood-based detection for normalizing flows - it's what they're designed for!

---

In [ ]:
# ==============================================================================
# PART 3: CycleGAN Training Example
# ==============================================================================

"""
Example usage for CycleGAN training

# 1. Download dataset (example: horse2zebra)
# You can use: apple2orange, summer2winter_yosemite, monet2photo, etc.
!wget https://people.eecs.berkeley.edu/~taesung_park/CycleGAN/datasets/horse2zebra.zip
!unzip horse2zebra.zip

# 2. Setup datasets
train_dataset_A = ImageDataset(
    root_dir='horse2zebra/trainA',
    transform=cyclegan_transform
)

train_dataset_B = ImageDataset(
    root_dir='horse2zebra/trainB',
    transform=cyclegan_transform
)

test_dataset_A = ImageDataset(
    root_dir='horse2zebra/testA',
    transform=cyclegan_transform
)

test_dataset_B = ImageDataset(
    root_dir='horse2zebra/testB',
    transform=cyclegan_transform
)

train_loader_A = DataLoader(train_dataset_A, batch_size=1, shuffle=True, num_workers=2)
train_loader_B = DataLoader(train_dataset_B, batch_size=1, shuffle=True, num_workers=2)
test_loader_A = DataLoader(test_dataset_A, batch_size=5, shuffle=False)
test_loader_B = DataLoader(test_dataset_B, batch_size=5, shuffle=False)

# 3. Visualize some samples
print("Sample images from domain A:")
sample_A = next(iter(test_loader_A))
visualize_samples(sample_A[:5])

print("Sample images from domain B:")
sample_B = next(iter(test_loader_B))
visualize_samples(sample_B[:5])

# 4. Initialize models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

G_AB = Generator(input_nc=3, output_nc=3, ngf=64, num_residual_blocks=9)
G_BA = Generator(input_nc=3, output_nc=3, ngf=64, num_residual_blocks=9)
D_A = Discriminator(input_nc=3, ndf=64)
D_B = Discriminator(input_nc=3, ndf=64)

print(f"G_AB parameters: {sum(p.numel() for p in G_AB.parameters()):,}")
print(f"G_BA parameters: {sum(p.numel() for p in G_BA.parameters()):,}")
print(f"D_A parameters: {sum(p.numel() for p in D_A.parameters()):,}")
print(f"D_B parameters: {sum(p.numel() for p in D_B.parameters()):,}")

# 5. Train the model
history = train_cyclegan(
    G_AB=G_AB,
    G_BA=G_BA,
    D_A=D_A,
    D_B=D_B,
    train_loader_A=train_loader_A,
    train_loader_B=train_loader_B,
    num_epochs=20,
    lr=0.0002,
    beta1=0.5,
    device=device
)

# 6. Plot training history
plot_training_history(history)

# 7. Test and visualize results at different epochs
print("\\nTesting at epoch 1 (early)...")
# Load checkpoint from epoch 1 and test
# test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device)

print("\\nTesting at epoch 10 (mid)...")
# Load checkpoint from epoch 10 and test
# test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device)

print("\\nTesting at epoch 20 (final)...")
test_cyclegan(G_AB, G_BA, test_loader_A, test_loader_B, device, num_samples=5)

# 8. Save models
torch.save(G_AB.state_dict(), 'G_AB_final.pth')
torch.save(G_BA.state_dict(), 'G_BA_final.pth')
torch.save(D_A.state_dict(), 'D_A_final.pth')
torch.save(D_B.state_dict(), 'D_B_final.pth')
"""
